In [5]:


text = """
LangChain 是一个用于构建 LLM 应用的框架，常用于开发 AI Agent、RAG（Retrieval-Augmented Generation）系统以及工作流编排。

很多开发者会把它当成“大模型时代的 Spring Framework”，因为它提供了 prompt management、memory、tool calling、retriever、agent orchestration 等模块化能力。

在检索增强生成场景中，开发者通常会将文本 chunk 存储到 vector database，例如 Chroma、FAISS、Milvus、Pinecone 或 Weaviate。

向量数据库（Vector Store）本质上保存的是 embedding 向量，而不是原始文本本身。用户查询时，系统会把 query 转换成 embedding，然后进行 semantic similarity search。

有些系统使用 cosine similarity，也有些使用 ANN（Approximate Nearest Neighbor）算法，比如 HNSW 或 IVF。

MultiQueryRetriever 是 LangChain 中一种提升召回率（recall）的技术。它会让 LLM 自动生成多个不同表达的查询语句，例如：

- “如何提高 RAG 检索效果”
- “怎样优化知识库召回”
- “improve retrieval quality in vector search”
- “增强 embedding 检索命中率的方法”

这些 query 会分别执行向量检索，然后对结果进行 merge 和 deduplicate。

这种方法特别适用于：
1. 用户问题表达模糊
2. chunk 中关键词不一致
3. 存在大量同义词
4. embedding 模型语义能力有限

例如，一篇文档中写的是“高并发缓存系统”，但用户问的是“redis scalability architecture”，单一 query 可能无法命中。

同样，“authentication” 和 “登录鉴权” 在某些 embedding 模型中距离并不近。

因此，多查询（multi query）能够从不同角度改写用户问题，提高 retrieval recall。

除了 MultiQueryRetriever，还有一些常见 RAG 优化技术：

- HyDE（Hypothetical Document Embeddings）
- Parent Document Retriever
- Contextual Compression Retriever
- Reranker
- BM25 + Vector Hybrid Search
- Self Query Retriever

在生产环境中，很多团队会结合 rerank model 与 hybrid retrieval 一起使用，以降低 hallucination 并提升 answer grounding。

LangGraph 是 LangChain 生态中的一个状态机工作流框架，用于构建 multi-agent systems、long-running workflow 和复杂推理链。

有些开发者认为：
- LangChain 更像应用层抽象
- LangGraph 更偏 orchestration engine
- DeepAgents 更像高级 agent runtime

在 AI Coding 场景中，RAG 常被用于：
- 代码库问答
- 文档检索
- API 搜索
- 企业知识库
- Wiki assistant

一个典型 pipeline 通常包括：

Document Loader → Text Splitter → Embedding Model → Vector Store → Retriever → LLM → Response Generator

如果 chunk size 太小，可能导致 context fragmentation；
如果 chunk 太大，又可能降低 embedding precision。

因此，chunk overlap、chunking strategy、metadata filtering 都会影响最终 retrieval performance。"""

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80
)
chunks = splitter.split_text(text)
print(len(chunks))

6


In [7]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama

model = OllamaEmbeddings(
    model="qwen3-embedding:8b",
    base_url="http://192.168.31.198:11434",
)
vector_store = Chroma(
    collection_name="c3-4",  # 给向量库起一个名字
    embedding_function=model,
    persist_directory="./chroma_db" # 指定数据存放的文件夹
)



In [8]:
# 删除同名 collection，并重新创建一个空 collection
vector_store.reset_collection()
vector_store.add_texts(chunks)

['3ee043f5-bb5f-476e-a238-ceb79ce80302',
 '7ab83c6f-9cdf-4a86-ad47-e68f197fbfc6',
 '840e8cad-b659-46ca-8668-d21dd74feea0',
 '56cafc59-0ebc-4d43-a750-72594a6d4bb8',
 '37fa7b47-edd6-4990-87f5-bc18800d1b56',
 '368092c8-460a-4cba-92e0-261a0ece0ca0']

## langchain 1.* 已经弃用MultiQueryRetriever
那就自己手动实现一个

In [9]:
from pydantic import BaseModel, Field

llm = ChatOllama(
    base_url="http://192.168.31.198:11434",
    model="qwen3:8b"
)
class Questions(BaseModel):
    """生成的三个问题"""
    question1: str = Field(description="生成的第1个问题")
    question2: str = Field(description="生成的第2个问题")
    question3: str = Field(description="生成的第3个问题")

query = "什么是langchain"
# prompt = f"""
# 原始问题是{query}, 帮我生成语义相近的三个问题
# """
prompt = f"""
你是一个 AI 语言模型助手，你的任务是生成给定用户问题的三个不同版本，以从在向量数据库中检索相关文档。
通过提供用户问题的多个视角，你的目标是帮助用户克服基于距离的相似性搜索的一些限制。
原始问题是：{query},
"""
llm_with_structured_output = llm.with_structured_output(
    Questions
)
queries = llm_with_structured_output.invoke(prompt)
print(queries)

question1='langchain的核心功能和架构是怎样的？' question2='在实际应用中，langchain如何帮助开发人员构建复杂的AI工作流？' question3='如何通过langchain优化AI工作流的效率？'


## 检索

### 查询结果融合 RRF

In [31]:
RRF_K = 60  # 论文实践中的常用默认值
content_rrf_scores = {}

for _, generated_query in queries:
    retrieved_documents = vector_store.similarity_search(generated_query)

    # 遍历每一篇文档
    for rank, document in enumerate(retrieved_documents, start=1):
        rrf_score = 1 / (RRF_K + rank)
        content = document.page_content
        content_rrf_scores[content] = content_rrf_scores.get(content, 0) + rrf_score

ranked_rrf_results = sorted(
    content_rrf_scores.items(),
    key=lambda item: item[1],
    reverse=True,
)
for content, rrf_score in ranked_rrf_results:
    print("=" * 20, rrf_score, "=" * 20)
    print(content[:50])



==================== 0.04918032786885246 ====================
LangChain 是一个用于构建 LLM 应用的框架，常用于开发 AI Agent、RAG（Ret
==================== 0.048131080389144903 ====================
有些开发者认为：
- LangChain 更像应用层抽象
- LangGraph 更偏 orches
==================== 0.04787506400409626 ====================
除了 MultiQueryRetriever，还有一些常见 RAG 优化技术：

- HyDE（Hy
==================== 0.046875 ====================
向量数据库（Vector Store）本质上保存的是 embedding 向量，而不是原始文本本身。


## 拼接问题和检索结果成一个 prompt

In [32]:

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
reference_text = ""
for content, _ in ranked_rrf_results:
    reference_text += content

prompt_all = f"""
用户的问题是:{query}
参考文本是:{reference_text}
"""
print(prompt_all)
chat_model = init_chat_model(
        base_url="http://192.168.31.198:11434",
        model="qwen3:8b",
        model_provider="ollama"
)
agent = create_agent(
    model=chat_model,
    system_prompt="你是一个专业的问答助手，请以用户给定的资料回答内容，如果用户给定的参考内容中不足以回答问题，则回答不知道。"
)
answer = agent.invoke({"messages": [
        {"role": "user", "content": prompt_all}
    ]})
print(answer)


用户的问题是:什么是langchain
参考文本是:LangChain 是一个用于构建 LLM 应用的框架，常用于开发 AI Agent、RAG（Retrieval-Augmented Generation）系统以及工作流编排。

很多开发者会把它当成“大模型时代的 Spring Framework”，因为它提供了 prompt management、memory、tool calling、retriever、agent orchestration 等模块化能力。

在检索增强生成场景中，开发者通常会将文本 chunk 存储到 vector database，例如 Chroma、FAISS、Milvus、Pinecone 或 Weaviate。有些开发者认为：
- LangChain 更像应用层抽象
- LangGraph 更偏 orchestration engine
- DeepAgents 更像高级 agent runtime

在 AI Coding 场景中，RAG 常被用于：
- 代码库问答
- 文档检索
- API 搜索
- 企业知识库
- Wiki assistant

一个典型 pipeline 通常包括：

Document Loader → Text Splitter → Embedding Model → Vector Store → Retriever → LLM → Response Generator

如果 chunk size 太小，可能导致 context fragmentation；
如果 chunk 太大，又可能降低 embedding precision。除了 MultiQueryRetriever，还有一些常见 RAG 优化技术：

- HyDE（Hypothetical Document Embeddings）
- Parent Document Retriever
- Contextual Compression Retriever
- Reranker
- BM25 + Vector Hybrid Search
- Self Query Retriever

在生产环境中，很多团队会结合 rerank model 与 hybrid retrieval 一起使用，以降低 hallucination 并提升 answer

In [33]:
print(answer["messages"][-1].content)

LangChain 是一个用于构建大语言模型（LLM）应用的框架，核心功能包括开发 AI Agent、检索增强生成（RAG）系统以及工作流编排。它被开发者类比为“大模型时代的 Spring Framework”，因其提供了模块化能力，如 prompt 管理、记忆（memory）、工具调用（tool calling）、检索器（retriever）和代理编排（agent orchestration）等。

在 RAG 场景中，LangChain 通过将文本分块存储到向量数据库（如 Chroma、FAISS 等）实现语义检索，并结合 LLM 生成答案。其典型流程包括文档加载、文本分割、向量存储、检索和生成响应等环节。为优化检索效果，LangChain 支持多种技术，如 MultiQueryRetriever（生成多组查询）、HyDE（假设文档嵌入）、reranker（重排序）等，以降低幻觉（hallucination）并提升答案的准确性。

此外，LangChain 生态中还包含 LangGraph 等工具，但 LangChain 本身更偏向应用层抽象，而 LangGraph 更侧重于状态机和复杂工作流的编排。
